## Prerequsite

In [ ]:
import asyncio
import nest_asyncio
# Patch the event loop so LlamaIndex async calls work inside Jupyter
nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv()

In [ ]:
import pandas as pd
import networkx as nx

from llama_index.core import Document, PropertyGraphIndex, Settings
from llama_index.core.graph_stores.types import (
    EntityNode, KG_NODES_KEY, KG_RELATIONS_KEY, Relation
)
from llama_index.core.graph_stores import SimplePropertyGraphStore
from llama_index.core.llms.llm import LLM
from llama_index.core.prompts import PromptTemplate
from llama_index.core.schema import TransformComponent, BaseNode
from llama_index.core.async_utils import run_jobs
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.llms.openai import OpenAI
from graspologic.partition import hierarchical_leiden

print("Imports ready")

In [ ]:
EXTRACTION_LLM = OpenAI(model="gpt-4o-mini", temperature=0)
QUERY_LLM      = OpenAI(model="gpt-4o",      temperature=0)

MAX_ARTICLES        = 50
MAX_PATHS_PER_CHUNK = 20
NUM_WORKERS         = 4
MAX_CLUSTER_SIZE    = 10
GRAPH_OUTPUT_FILE   = "ai_copyright_graph.html"


Settings.llm = EXTRACTION_LLM

print(f"✅ Using {EXTRACTION_LLM.model} for extraction, {QUERY_LLM.model} for querying")

In [ ]:
ENTITY_TYPES = [
    "ORGANIZATION",
    "PERSON",
    "LEGISLATION",
    "LEGAL_CASE",
    "CONCEPT",
    "GOVERNMENT",
    "AI_SYSTEM",
]

RELATION_TYPES = [
    "FILED_AGAINST",
    "DEFENDANT_IN",
    "REGULATES",
    "ADVOCATES_FOR",
    "TRAINED_ON",
    "PART_OF",
    "REFERENCES",
    "OPPOSES",
]

print("✅ Ontology:")
print(f"   Entity types:       {ENTITY_TYPES}")
print(f"   Relationship types: {RELATION_TYPES}")

In [ ]:
_entity_types = ", ".join(ENTITY_TYPES)
_relation_types = ", ".join(RELATION_TYPES)

EXTRACT_TMPL = f"""
-Goal-
Given a news article about AI copyright, governance, or intellectual property,
identify all entities mentioned in the article and their relationships.

Extract up to {{max_knowledge_triplets}} entity-relation triplets.

-Allowed Entity Types-
{_entity_types}

-Allowed Relationship Types-
{_relation_types}

-Steps-
1. Identify ALL entities. For each entity extract:
   - name: Name of the entity, capitalized
   - type: One of the allowed entity types above
   - description: A brief description of the entity and its role in AI copyright/governance

2. Identify relationships between entities. For each pair extract:
   - source: name of the source entity
   - target: name of the target entity
   - relation: one of the allowed relationship types above
   - description: a sentence explaining why and how these entities are related

-Real Data-
######################
text: {{text}}
######################
"""

print("✅ Extraction prompt ready")
print(f"\nPreview (first 300 chars):\n{EXTRACT_TMPL[:300]}...")

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import Literal, List

EntityType = Literal[
    "ORGANIZATION",
    "PERSON",
    "LEGISLATION",
    "LEGAL_CASE",
    "CONCEPT",
    "GOVERNMENT",
    "AI_SYSTEM",
]

RelationType = Literal[
    "FILED_AGAINST",
    "DEFENDANT_IN",
    "REGULATES",
    "ADVOCATES_FOR",
    "TRAINED_ON",
    "PART_OF",
    "REFERENCES",
    "OPPOSES",
]

class ExtractedEntity(BaseModel):
    name: str = Field(description="Name of the entity, captitalized")
    type: EntityType = Field(description="One of the allowed entity types")
    description: str = Field(description="Brief description of the entity and its role")

class ExtractedRelationship(BaseModel):
    source: str = Field(description="Name of the source entity")
    target: str = Field(description="Name of the target entity")
    relation: RelationType = Field(description="One of the allowed relationship types")
    description: str = Field(description="Sentence explaining the relationship")

class ExtractionResult(BaseModel):
    entities: List[ExtractedEntity] = Field(default_factory=list)
    relationships: List[ExtractedRelationship] = Field(default_factory=list)

print("Pydantic extraction models defined")




## GraphRAG Extractor

In [ ]:
class GraphRAGExtractor(TransformComponent):
    """
    Extracts entities and relationships WITH descriptions from each text chunk(using LLM).

    Uses Pydantic structured output (via OpenAI function calling) to guarantee
    that every entity has a valid type and every relationship uses an allowed label.

    Data flow for a single chunk:
      Raw text
        → LLM (astructured_predict)
        → ExtractionResult(
              entities=[
                  ExtractedEntity(name="OpenAI", type="ORGANIZATION", description="AI lab..."),
                  ExtractedEntity(name="NYT v. OpenAI", type="LEGAL_CASE", description="Copyright lawsuit..."),
              ],
              relationships=[
                  ExtractedRelationship(source="OpenAI", target="NYT v. OpenAI",
                                        relation="DEFENDANT_IN", description="OpenAI is named defendant..."),
              ]
          )
        → EntityNode(name="OpenAI", label="ORGANIZATION", properties={...})
        → EntityNode(name="NYT v. OpenAI", label="LEGAL_CASE", properties={...})
        → Relation(label="DEFENDANT_IN", source_id=<OpenAI id>, target_id=<NYT v. OpenAI id>)
    """

    llm: LLM = Field(default_factory=lambda: Settings.llm)
    instruction_prompt: PromptTemplate = Field(default_factory=lambda: PromptTemplate(EXTRACT_TMPL))
    num_workers: int = 4
    max_paths_per_chunk: int = 20

    @field_validator("instruction_prompt", mode="before")
    @classmethod
    def convert_to_prompt_template(cls, v):
        return PromptTemplate(v) if isinstance(v, str) else v
    
    def __call__(self, nodes, show_progress=False, **kwargs):
        return asyncio.run(self._call(nodes, show_progress=show_progress, **kwargs))
    
    async def _call(self, nodes, show_progress=False, **kwargs):
        jobs = [self._extract(node) for node in nodes]
        return await run_jobs(
            jobs,
            workers = self.num_workers,
            show_progress = show_progress,
            desc="Extracting..."
        )


    async def _extract(self, node: BaseNode) -> BaseNode:
        text = node.get_content(metadata_mode="llm")

        try:
            result = await self.llm.astructured_predict(
                ExtractionResult,
                self.instruction_prompt,
                text = text,
                max_knowledge_triplets = self.max_paths_per_chunk
            )
            entities = result.entities
            relationships = result.relationships
        except Exception as e:
            print(f"Extraction error: {e}")
            entities, relationships = [], []
        
        existing_nodes     = node.metadata.pop(KG_NODES_KEY, [])
        existing_relations = node.metadata.pop(KG_RELATIONS_KEY, [])
        base_metadata      = node.metadata.copy()

        existing_nodes += [
            EntityNode(
                name = entity.name,
                label = entity.type,
                properties = {**base_metadata, "entity_description": entity.description},
            )
            for entity in entities
        ]

        entity_lookup = {e.name: e.type for e in entities}

        for rel in relationships:
            source_node = EntityNode(
                name = rel.source,
                label = entity_lookup.get(rel.source, "ENTITY"),
                properties=base_metadata,
            )
            target_node = EntityNode(
                name = rel.target,
                label = entity_lookup.get(rel.target, "ENTITY"),
            )

            if rel.source not in entity_lookup:
                existing_nodes.append(source_node)
            if rel.target not in entity_lookup:
                existing_nodes.append(target_node)
            
            existing_relations.append(Relation(
                label = rel.relation,
                source_id = source_node.id,
                target_id = target_node.id,
                properties = {**base_metadata, "relationship_description": rel.description},
            ))

        node.metadata[KG_NODES_KEY]     = existing_nodes
        node.metadata[KG_RELATIONS_KEY] = existing_relations
        return node


In [ ]:
class GraphRAGStore(SimplePropertyGraphStore):
    """
    Generate communities from graph that extracted by LLM.
    """

    # Leiden community detection
    community_summaries: dict = {}
    
    def build_communities(self):
        print("Running community detection...")
        nx_graph = self._to_networks()

        if not nx_graph.nodes:
            print("Graph is empty - no communities to detect")
            return
        
        print(f"Graph has {nx_graph.number_of_edges()} nodes, {nx_graph.number_of_edges} edges")

        clusters = hierarchical_leiden(nx_graph, max_cluster_size=MAX_CLUSTER_SIZE)
        num_communities = len(set(c.cluster for c in clusters))
        print(f"Found {num_communities} communities")

        community_info = self._collect_community_info(nx_graph, clusters)

        self._agent_summaries(community_info)
        print(f"{len(self.community_summaries)} community summaries generated")

    def _to_networks(self) -> nx.Graph:
        nx_graph = nx.Graph()

        for node in self.graph.nodes.values():
            if isinstance(node, EntityNode):
                nx_graph.add_node(node.id)
            # else:
            #     print(f"WARM: {node} is not EntityNode")
            
        for rel in self.graph.relations.values():
            if rel.source_id in nx_graph and rel.target_id in nx_graph:
                nx_graph.add_edge(
                    rel.source_id,
                    rel.target_id,
                    relationship = rel.label,
                )
            # else:
            #     print(f"WARM: {rel} is not following the requirements")
        
        return nx_graph

    def _collect_community_info(self, nx_graph, clusters) -> dict:

        node_to_cluster = {item.node: item.cluster for item in clusters}

        node_details = {}
        for node in self.graph.nodes.values():
            if not isinstance(node, EntityNode):
                continue
            node_details[node.id] = {
                "name": node.name,
                "type": node.label,
                "description": node.properties.get("entity_description", ""),
            }

        community_info = {}
        for item in clusters:
            cid, nid = item.cluster, item.node 
            print(f"check: {cid} - {nid}")
            community_info.setdefault(cid, {"entities": [], "relationships": []})

            if nid in node_details:
                community_info[cid]["entities"].append(node_details[nid])
            
            for neighbor in nx_graph.neighbors(nid):
                if node_to_cluster.get(neighbor) == cid:
                    edge = nx_graph.get_edge_data(nid, neighbor)
                    rel  = edge.get("relationship", "RELATED") if edge else "RELATED"
                    desc = edge.get("description",  "")        if edge else ""

                    src_name = node_details.get(nid, {}).get("name", nid)
                    tar_name = node_details.get(neighbor, {}).get("name", neighbor)

                    entry = f"{src_name} --[{rel}]--> {tar_name}"
                    if desc:
                        entry += f" ({desc})"
                    
                    community_info[cid]["relationships"].append(entry)
        
        return community_info
    
    def _agent_summaries(self, community_info):
        
        for community_id, data in community_info.items():
            if not data["relationships"] and not data["entities"]:
                continue

            entities_text = "\n".join([
                f"- {e['name']} ({e['type']}): e{e['description']}"
                for e in data["entities"] if e.get("name")
            ])

            relationships_text = "\n".join(sorted(set(data["relationships"])))


            prompt = f"""You are analysing a cluster of entities from news articles about
            AI copyright, governance, and intellectual property.

            Entities in this cluster:
            {entities_text}

            Relationships:
            {relationships_text}

            Write a concise briefing (3-5 sentences) that:
            1. Identifies the main organizations, people, legal cases, or topics in this cluster
            2. Explains how they are connected and why — including legal or regulatory context
            3. Highlights any disputes, lawsuits, policy positions, or tensions
            4. Notes anything particularly relevant for understanding AI copyright or governance

            Briefing:"""

            response = EXTRACTION_LLM.complete(prompt)
            self.community_summaries[community_id] = response.text
            print(f"  Community {community_id}: {response.text[:100]}...")
        
    def get_community_summaries(self) -> dict:
        return self.community_summaries
    
print("GraphRAGStore defined")
        


In [ ]:
class GraphRAGQueryEngine(CustomQueryEngine):
    """
    Query community summary.
    """
    graph_store: GraphRAGStore
    llm: LLM

    def custom_query(self, query: str) -> str:
        summaries = self.graph_store.get_community_summaries()

        if not summaries:
            return "No community summaries found. Run graph_store.build_communities() first."
        
        communite_answers = [
            self._answer_from_community(summary, query)
            for summary in summaries.values()
        ]

        relavant_answers = [a for a in communite_answers if a.strip()]

        if not relavant_answers:
            return "I don't have enough information in the knowledge graph to answer that question"
        
        return self._aggregate(relavant_answers, self.query)
    
    def _answer_from_community(self, summary: str, query: str) -> str:
        prompt = (
            f"Community summary: \n{summary}\n\n"
            f"Question: {query}\n\n"
            f"If this summary contains information relevant to the question, answer it. "
            f"If not relevant, reply exactly: 'No relevant information.'\n\n"
            f"Answer:"
        )
        response = EXTRACTION_LLM.complete(prompt)
        text = response.text.strip()
        return "" if "no relevant information" in text.lower() else text

    def _aggregate(self, answers: List[str], query: str) -> str:
        combined = "\n\n---\n\n".join(answers)

        prompt = (
            f"You have received answers from multiple knowledge graph communities about this question:\n\n"
            f"Question: {query}\n\n"
            f"Community answers:\n{combined}\n\n"
            f"Synthesise these into a single, clear, well-structured final answer. "
            f"Remove redundancy, keep all important details, and ensure the answer "
            f"directly addresses the question.\n\n"
            f"Final Answer:"
        )
        
        return self.llm.complete(prompt).text

print("GraphRAGQueryEngine defined")


In [ ]:
df = pd.read_csv("dataset.csv")
print(f"Number of articles: {len(df)}")
df.head()

In [ ]:
nodes = [
    Document(
        text=str(row["full_text"]),
        metadata={
            "title": str(row.get("title", "")),
            "source": str(row.get("source", "")),
            "date": str(row.get("date", "")),
        }
    )
    for _, row in df.iterrows()
]
print(f"Created {len(nodes)} LlamaIndex documents, stored as nodes")

In [ ]:
print(nodes[1].text)

In [ ]:
extractor = GraphRAGExtractor(
    llm=EXTRACTION_LLM,
    extract_prompt=EXTRACT_TMPL,
    max_paths_per_chunk=MAX_PATHS_PER_CHUNK,
    num_workers=NUM_WORKERS,
)

graph_store = GraphRAGStore()

print("Building knowledge graph... (this may take a few minutes)")

graph_rag_index = PropertyGraphIndex(
    nodes=nodes,
    kg_extractors=[extractor],
    property_graph_store=graph_store,
    embed_kg_nodes=False,             # we don't need vector similarity search for this GraphRAG pipeline
    show_progress=True,
)


In [ ]:
# import copy

# sample = nodes[3]

# print("=== Title ===")
# print(sample.metadata.get("title", "untitled"))
# print()
# print("=== Raw text ===")
# print(sample.text)
# print()

# extracted = await extractor._extract(copy.deepcopy(sample))

# print("=== Extracted entities ===")
# for node in extracted.metadata.get(KG_NODES_KEY, []):
#     print(f"  [{node.label}] {node.name}")
#     print(f"    {node.properties.get('entity_description', '')}")

# print()
# print("=== Extracted relationships ===")
# id_to_name = {n.id: n.name for n in extracted.metadata.get(KG_NODES_KEY, [])}
# for rel in extracted.metadata.get(KG_RELATIONS_KEY, []):
#     src = id_to_name.get(rel.source_id, rel.source_id)
#     tgt = id_to_name.get(rel.target_id, rel.target_id)
#     print(f"  {src} --[{rel.label}]--> {tgt}")

In [ ]:
# inspect_name = "U.S. CONGRESS"

# node = next(
#     (n for n in graph_store.graph.nodes.values()
#      if isinstance(n, EntityNode) and n.name == inspect_name),
#     None
# )

# print(f"Node:   {node.name!r}  label={node.label!r}")
# print(f"Source: {node.properties.get('source', 'N/A')}")
# print(f"Title:  {node.properties.get('title', 'N/A')}")
# print()

# title = node.properties.get("title")
# article = next((n for n in nodes if n.metadata.get("title") == title), None)
# if article:
#     print("=== Article text ===")
#     print(article.text)
# print()

# relations = [
#     r for r in graph_store.graph.relations.values()
#     if r.source_id == node.id or r.target_id == node.id
# ]
# node_index = {
#     n.id: n.name for n in graph_store.graph.nodes.values()
#     if isinstance(n, EntityNode)
# }
# print(f"Relations ({len(relations)}):")
# for r in relations:
#     src = node_index.get(r.source_id, r.source_id)
#     tgt = node_index.get(r.target_id, r.target_id)
#     print(f"  {src} --[{r.label}]--> {tgt}")

In [ ]:
graph_store.build_communities()

summaries = graph_store.get_community_summaries()
print(f"\n{len(summaries)} community summaries ready for querying")

In [ ]:
import json
import pathlib


def export_graph_data(graph_store: GraphRAGStore, output_file: str = "graph_data.json"):
    """
    Export the knowledge graph to a JSON file compatible with the D3.js template.
    Run this once after graph_store.build_communities().

    Saves to disk so you can re-run the visualization without reprocessing
    the full pipeline — which is expensive.
    """

    nx_graph = graph_store._to_networks()

    # ── Build node metadata lookup ─────────────────────────────────────────
    node_meta = {}
    for node in graph_store.graph.nodes.values():
        if isinstance(node, EntityNode):
            node_meta[node.id] = {
                "id":          node.id,
                "label":       node.name,
                "type":        node.label if node.label else "OTHER",
                "description": node.properties.get("entity_description", ""),
            }

    # ── Nodes — only those present in the NetworkX graph ──────────────────
    nodes_data = [
        node_meta[n] for n in nx_graph.nodes() if n in node_meta
    ]

    # ── Links ──────────────────────────────────────────────────────────────
    links_data = []
    for source, target, data in nx_graph.edges(data=True):
        links_data.append({
            "source":      source,
            "target":      target,
            "label":       data.get("relationship", ""),
            "description": data.get("description",  ""),
        })

    # ── Assemble and write ─────────────────────────────────────────────────
    graph_data = {
        "nodes":       nodes_data,
        "links":       links_data,
        "communities": len(graph_store.get_community_summaries()),
    }

    pathlib.Path(output_file).write_text(json.dumps(graph_data, indent=2))

    print(f"✅ Graph data exported to '{output_file}'")
    print(f"   Nodes:       {len(nodes_data)}")
    print(f"   Edges:       {len(links_data)}")
    print(f"   Communities: {graph_data['communities']}")

    return graph_data


def visualize_graph(
    graph_store: GraphRAGStore = None,
    graph_data_file: str = "graph_data.json",
    template_file: str = "graph_template.html",
    output_file: str = GRAPH_OUTPUT_FILE,
):
    """
    Generate a D3.js interactive knowledge graph visualization.

    Can be called in two ways:

    1. Pass graph_store directly — exports fresh data then builds the HTML:
         visualize_graph(graph_store=graph_store)

    2. Load from a previously exported JSON file — no pipeline re-run needed:
         visualize_graph()  # loads graph_data.json automatically

    Features:
    - Color-coded nodes by entity type (from ontology)
    - Node size scales with degree (more connections = bigger)
    - Click a node to highlight its neighbourhood, dim everything else
    - Sidebar legend — click any entity type to toggle its visibility
    - Search bar to find and highlight any node by name
    - Sliders for link distance, charge strength, and edge label threshold
    - Hover tooltips with entity name, type, description, and connection count
    - Stats header showing total nodes, edges, and communities
    """

    # ── Get graph data ─────────────────────────────────────────────────────
    if graph_store is not None:
        # Export fresh data from the graph store and save to disk
        graph_data = export_graph_data(graph_store, graph_data_file)
    else:
        # Load from a previously saved JSON file
        data_path = pathlib.Path(graph_data_file)
        if not data_path.exists():
            print(f"⚠️  '{graph_data_file}' not found.")
            print(f"   Run export_graph_data(graph_store) first, or pass graph_store directly.")
            return
        graph_data = json.loads(data_path.read_text())

        print(f"✅ Loaded graph data from '{graph_data_file}'")
        print(f"   Nodes: {len(graph_data['nodes'])}  |  "
              f"Edges: {len(graph_data['links'])}  |  "
              f"Communities: {graph_data.get('communities', '—')}")

    # ── Load HTML template ─────────────────────────────────────────────────
    template_path = pathlib.Path(template_file)
    if not template_path.exists():
        print(f"⚠️  '{template_file}' not found.")
        print(f"   Make sure graph_template.html is in the same folder as this notebook.")
        return

    # ── Inject data into template and write output ─────────────────────────
    html = template_path.read_text()
    html = html.replace("GRAPH_DATA_PLACEHOLDER", json.dumps(graph_data))
    pathlib.Path(output_file).write_text(html)

    print(f"\n✅ Visualization saved to '{output_file}'")
    print(f"   Open it in your browser to explore.")


# Export data and build visualization from the graph store
visualize_graph(graph_store=graph_store)

In [ ]:
query_engine = GraphRAGQueryEngine(
    graph_store=graph_store,
    llm=QUERY_LLM,  # gpt-4o for final synthesis
)

print("Query engine ready")

In [ ]:
q1 = "What did Beijing Internet Court make?"
print(f"Query: {q1}")
print("=" * 70)
print(query_engine.custom_query(q1))

In [ ]:
q1 = "What are the main legal arguments being made around AI copyright and training data?"
print(f"Query: {q1}")
print("=" * 70)
print(query_engine.custom_query(q1))

In [ ]:
q2 = "Which companies are involved in AI copyright or governance disputes, and what are their positions?"
print(f"Query: {q2}")
print("=" * 70)
print(query_engine.custom_query(q2))

In [ ]:
q3 = "How are different governments (e.g. EU, US, and UK) approaching AI governance?"
print(f"Query: {q3}")
print("=" * 70)
print(query_engine.custom_query(q3))

In [ ]:
your_question = "What is the role of fair use in AI copyright disputes?"

print(f"Query: {your_question}")
print("=" * 70)
print(query_engine.custom_query(your_question))

In [ ]:
summary = query_engine.graph_store.get_community_summaries()
for key, value in summary.items():
    print(key)
    print(value)